In [1]:
from pathlib import Path

# Different user's home
PHISAT_PATH = Path("/home/heo/projects/py-q-nas/data/burnt_area_dataset/burned.zarr")

# Your project
IRIDE_PATH = Path("/root/projects/iride_onboard-burnscar-mapper/processed")

# Both can be read in the same script
print(f"PhiSat exists: {PHISAT_PATH.exists()}")
print(f"IRIDE exists: {IRIDE_PATH.exists()}")

PhiSat exists: True
IRIDE exists: True


In [3]:
# training/compute_stats_dask.py (FIXED)

import dask.array as da
import zarr
import rasterio
import numpy as np
from pathlib import Path
from rasterio.windows import Window

print("="*80)
print("STATS: LAZY LOADING + VALID PIXELS ONLY")
print("="*80)

# ============================================================================
# PHISAT STATS (Dask lazy, no memory crash)
# ============================================================================

print("\nPhiSat-2:")
phisat_root = zarr.open("/home/heo/projects/py-q-nas/data/burnt_area_dataset/burned.zarr/trainval", mode='r')
sample_ids = sorted(phisat_root.keys())[:100]  # Sample 100

# Build lazy array without loading
phisat_arrays = []
for sample_id in sample_ids:
    arr = phisat_root[sample_id]['img']  # zarr array
    # Convert zarr Array to dask Array (lazy!)
    darr = da.from_array(arr, chunks=(7, 128, 128))  # chunks for efficiency
    phisat_arrays.append(darr)

# Stack all samples along new axis
phisat_stack = da.stack(phisat_arrays, axis=0)  # (100, 7, 256, 256)

print(f"  Shape (lazy): {phisat_stack.shape}")
print(f"  Computing stats (only 100 samples)...")

# Compute mean/std across batch and spatial dims
phisat_mean = phisat_stack.mean(axis=(0, 2, 3)).compute()  # (7,)
phisat_std = phisat_stack.std(axis=(0, 2, 3)).compute()    # (7,)

print(f"  Mean per band: {phisat_mean}")
print(f"  Std per band:  {phisat_std}")
print(f"  Global: μ={phisat_mean.mean():.4f}  σ={phisat_std.mean():.4f}")

# ============================================================================
# IRIDE STATS (chunk by chunk, only VALID pixels)
# ============================================================================

print("\nIRIDE HEO (valid pixels only):")

iride_root = Path("/root/projects/iride_onboard-burnscar-mapper/processed")
stack_files = sorted(iride_root.glob("*_stack.tif"))
mask_files = sorted(iride_root.glob("*_mask.tif"))

# Accumulate per-band stats
band_stats = {b: {'sum': 0.0, 'sum_sq': 0.0, 'count': 0} for b in range(7)}

for stack_f, mask_f in zip(stack_files, mask_files):
    print(f"  {stack_f.name}...", end=" ", flush=True)
    
    with rasterio.open(stack_f) as si, rasterio.open(mask_f) as mi:
        H, W = si.height, si.width
        
        # Read in chunks to avoid memory spike
        chunk_size = 512
        
        for row in range(0, H, chunk_size):
            for col in range(0, W, chunk_size):
                win = Window(col, row, 
                            min(chunk_size, W - col),
                            min(chunk_size, H - row))
                
                # Read only this chunk
                img_chunk = si.read(window=win).astype(np.float32)  # (7, chunk_h, chunk_w)
                mask_chunk = mi.read(1, window=win)  # (chunk_h, chunk_w)
                
                # Filter: valid = not nodata (mask 0 or 255, but also exclude zero DN)
                valid = (mask_chunk > 0) & (mask_chunk < 255)
                
                # Also exclude pixels with zero DN (the corners)
                for b in range(7):
                    valid_band = valid & (img_chunk[b] > 0)
                    band_valid = img_chunk[b][valid_band]
                    
                    if band_valid.size > 0:
                        band_stats[b]['sum'] += band_valid.sum()
                        band_stats[b]['sum_sq'] += (band_valid ** 2).sum()
                        band_stats[b]['count'] += band_valid.size
    
    print("ok")

# Compute mean/std from accumulated sums
total_valid = band_stats[0]['count']
print(f"\n  Computed from {total_valid:,} valid pixels (excluding nodata & zero DN)")

iride_means = []
iride_stds = []

for b in range(7):
    count = band_stats[b]['count']
    if count == 0:
        print(f"  B{b}: NO VALID PIXELS!")
        iride_means.append(0)
        iride_stds.append(0)
        continue
    
    mean = band_stats[b]['sum'] / count
    variance = (band_stats[b]['sum_sq'] / count) - (mean ** 2)
    std = np.sqrt(max(variance, 0))
    
    iride_means.append(mean)
    iride_stds.append(std)
    print(f"  B{b}: μ={mean:.1f}  σ={std:.1f}")

iride_mean_global = np.mean(iride_means)
iride_std_global = np.mean(iride_stds)

print(f"  Global: μ={iride_mean_global:.1f}  σ={iride_std_global:.1f}")

# ============================================================================
# NORMALIZE & COMPARE
# ============================================================================

print("\n" + "="*80)
print("NORMALIZATION PARAMS")
print("="*80)

phisat_global_mean = float(phisat_mean.mean())
phisat_global_std = float(phisat_std.mean())

iride_max_dn = 1072  # from your data inspection
iride_normalized_mean = iride_mean_global / iride_max_dn
iride_normalized_std = iride_std_global / iride_max_dn

print(f"\nPhiSat-2 (already normalized [0,1]):")
print(f"  mean = {phisat_global_mean:.4f}")
print(f"  std  = {phisat_global_std:.4f}")

print(f"\nIRIDE HEO (raw DN [0-{iride_max_dn}], divide by {iride_max_dn}):")
print(f"  raw mean = {iride_mean_global:.1f}")
print(f"  raw std  = {iride_std_global:.1f}")
print(f"  normalized mean = {iride_normalized_mean:.4f}")
print(f"  normalized std  = {iride_normalized_std:.4f}")

print(f"\nDifference (PhiSat vs IRIDE normalized):")
print(f"  Δmean = {abs(phisat_global_mean - iride_normalized_mean):.4f}")
print(f"  Δstd  = {abs(phisat_global_std - iride_normalized_std):.4f}")

print(f"\n✓ NORMALIZATION CODE FOR TRAINING:")
print(f"  PhiSat: img = img  (already [0,1])")
print(f"  IRIDE:  img = np.clip(img / {iride_max_dn}, 0, 1)")

STATS: LAZY LOADING + VALID PIXELS ONLY

PhiSat-2:
  Shape (lazy): (100, 7, 256, 256)
  Computing stats (only 100 samples)...
  Mean per band: [0.41771156 0.40172157 0.36337602 0.41776076 0.3784842  0.40409663
 0.42484605]
  Std per band:  [0.13076493 0.12621345 0.15798526 0.22443141 0.16869897 0.20637675
 0.22436951]
  Global: μ=0.4011  σ=0.1770

IRIDE HEO (valid pixels only):
  IMH01_1CST__OPT8_20250709T100945_20250709T100947_20260330T172935_02623______O_A02_stack.tif... 

/tmp/ipykernel_3354145/525646745.py:74: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  mask_chunk = mi.read(1, window=win)  # (chunk_h, chunk_w)


ok
  IMH02_1CST__OPT8_20251115T104743_20251115T104745_20260511T102716_02159______O_A02_stack.tif... ok
  IMH02_1CST__OPT8_20260711T125448_20260711T125450_20260712T174910_05716______O_A02_stack.tif... ok
  IMH02_1CST__OPT8_20260711T125501_20260711T125504_20260712T175013_05716______O_A02_stack.tif... ok
  IMH05_1CST__OPT8_20260622T122713_20260622T122715_20260624T163803_05433______O_A02_stack.tif... ok
  IMH09_1CST__OPT8_20260722T124549_20260722T124552_20260722T155456_05875______O_A02_stack.tif... ok
  IMH09_1CST__OPT8_20260722T124551_20260722T124553_20260722T155532_05875______O_A02_stack.tif... ok

  Computed from 1,849,355 valid pixels (excluding nodata & zero DN)
  B0: μ=429.7  σ=74.7
  B1: μ=400.3  σ=93.5
  B2: μ=401.3  σ=134.2
  B3: μ=437.1  σ=179.1
  B4: μ=590.9  σ=231.7
  B5: μ=689.9  σ=267.7
  B6: μ=712.1  σ=242.7
  Global: μ=523.0  σ=174.8

NORMALIZATION PARAMS

PhiSat-2 (already normalized [0,1]):
  mean = 0.4011
  std  = 0.1770

IRIDE HEO (raw DN [0-1072], divide by 1072):
  ra

In [7]:
# training/check_normalized_stats.py

import numpy as np

print("="*80)
print("ACTUAL STATISTICS AFTER NORMALIZATION")
print("="*80)

# Raw data stats (computed earlier)
PHISAT_MEAN = 0.4011
PHISAT_STD = 0.1770

IRIDE_MEAN_DN = 523.0
IRIDE_STD_DN = 174.8
IRIDE_MAX_DN = 1072

# ============================================================================
# SCENARIO 1: Apply PHISAT normalization to PHISAT data
# ============================================================================
print("\nSCENARIO 1: PhiSat normalized with PhiSat params")
print("-" * 80)
print(f"Input (raw reflectance):  μ={PHISAT_MEAN:.4f}  σ={PHISAT_STD:.4f}")

# z = (x - mean) / std
# By definition of z-score: mean(z) = 0, std(z) = 1
phisat_norm_mean = 0.0
phisat_norm_std = 1.0

print(f"After z-score:             μ={phisat_norm_mean:.4f}  σ={phisat_norm_std:.4f}")
print(f"  (by definition of z-score)")

# ============================================================================
# SCENARIO 2: Apply IRIDE normalization to IRIDE data
# ============================================================================
print("\nSCENARIO 2: IRIDE normalized with IRIDE params")
print("-" * 80)
print(f"Input (raw DN):            μ={IRIDE_MEAN_DN:.1f}  σ={IRIDE_STD_DN:.1f}")

# First: convert to reflectance
iride_refl_mean = IRIDE_MEAN_DN / IRIDE_MAX_DN
iride_refl_std = IRIDE_STD_DN / IRIDE_MAX_DN

print(f"After DN→reflectance:      μ={iride_refl_mean:.4f}  σ={iride_refl_std:.4f}")

# Then: z-score with IRIDE params (but using raw DN values)
# z = (x_dn - 523.0) / 174.8
# mean(z) = 0, std(z) = 1
iride_norm_mean = 0.0
iride_norm_std = 1.0

print(f"After z-score:             μ={iride_norm_mean:.4f}  σ={iride_norm_std:.4f}")
print(f"  (by definition of z-score)")

# ============================================================================
# SCENARIO 3 (CRITICAL): Apply PHISAT normalization to IRIDE data
# ============================================================================
print("\nSCENARIO 3 (CROSS-DOMAIN): IRIDE data with PhiSat params")
print("-" * 80)
print(f"Input (raw DN):            μ={IRIDE_MEAN_DN:.1f}  σ={IRIDE_STD_DN:.1f}")

# Step 1: Convert to reflectance
print(f"Step 1 - DN→reflectance:   μ={iride_refl_mean:.4f}  σ={iride_refl_std:.4f}")

# Step 2: Apply PhiSat's z-score
# z = (x - 0.4011) / 0.1770
# mean(z) = (mean(x) - 0.4011) / 0.1770
# std(z) = std(x) / 0.1770

iride_with_phisat_mean = (iride_refl_mean - PHISAT_MEAN) / PHISAT_STD
iride_with_phisat_std = iride_refl_std / PHISAT_STD

print(f"Step 2 - Apply PhiSat z-score:")
print(f"  z = (x - {PHISAT_MEAN:.4f}) / {PHISAT_STD:.4f}")
print(f"  μ = ({iride_refl_mean:.4f} - {PHISAT_MEAN:.4f}) / {PHISAT_STD:.4f} = {iride_with_phisat_mean:.4f}")
print(f"  σ = {iride_refl_std:.4f} / {PHISAT_STD:.4f} = {iride_with_phisat_std:.4f}")

print(f"After PhiSat norm:         μ={iride_with_phisat_mean:.4f}  σ={iride_with_phisat_std:.4f}")

# ============================================================================
# SUMMARY: Is PhiSat pre-training viable?
# ============================================================================

print("\n" + "="*80)
print("DOMAIN SHIFT ANALYSIS")
print("="*80)

print(f"\nPhiSat (with PhiSat norm):     μ=0.0000  σ=1.0000")
print(f"IRIDE (with IRIDE norm):       μ=0.0000  σ=1.0000")
print(f"IRIDE (with PhiSat norm):      μ={iride_with_phisat_mean:.4f}  σ={iride_with_phisat_std:.4f}")

print(f"\nDomain shift magnitude:")
print(f"  Δmean = {abs(iride_with_phisat_mean):.4f}  ← mean is SHIFTED by this much")
print(f"  Δstd  = {abs(1.0 - iride_with_phisat_std):.4f}  ← std is DIFFERENT by this much")

if abs(iride_with_phisat_mean) < 0.5 and abs(1.0 - iride_with_phisat_std) < 0.2:
    print(f"\n✓ SMALL domain shift — PhiSat pre-training is VIABLE")
    print(f"  The model should transfer reasonably well")
else:
    print(f"\n⚠ LARGE domain shift — PhiSat pre-training may struggle")
    print(f"  Consider:")
    print(f"  - Training only on IRIDE (no PhiSat)")
    print(f"  - Using different IRIDE normalization")
    print(f"  - Domain adaptation techniques")

print(f"\n" + "="*80)
print("RECOMMENDATION")
print("="*80)
print(f"\nUse separate normalization per sensor:")
print(f"  PhiSat: z = (x - {PHISAT_MEAN:.4f}) / {PHISAT_STD:.4f}")
print(f"  IRIDE:  z = (x - {IRIDE_MEAN_DN:.1f}) / {IRIDE_STD_DN:.1f}")
print(f"\nBoth will have mean=0, std=1 in their own space")
print(f"Model learns on both → robust to sensor differences")

ACTUAL STATISTICS AFTER NORMALIZATION

SCENARIO 1: PhiSat normalized with PhiSat params
--------------------------------------------------------------------------------
Input (raw reflectance):  μ=0.4011  σ=0.1770
After z-score:             μ=0.0000  σ=1.0000
  (by definition of z-score)

SCENARIO 2: IRIDE normalized with IRIDE params
--------------------------------------------------------------------------------
Input (raw DN):            μ=523.0  σ=174.8
After DN→reflectance:      μ=0.4879  σ=0.1631
After z-score:             μ=0.0000  σ=1.0000
  (by definition of z-score)

SCENARIO 3 (CROSS-DOMAIN): IRIDE data with PhiSat params
--------------------------------------------------------------------------------
Input (raw DN):            μ=523.0  σ=174.8
Step 1 - DN→reflectance:   μ=0.4879  σ=0.1631
Step 2 - Apply PhiSat z-score:
  z = (x - 0.4011) / 0.1770
  μ = (0.4879 - 0.4011) / 0.1770 = 0.4902
  σ = 0.1631 / 0.1770 = 0.9212
After PhiSat norm:         μ=0.4902  σ=0.9212

DOMAIN SH